In [ ]:
from grids import Cell, generate_network,  convert_coords_to_2d, SpaceType, Partition,Splits
from utils import evaluate_kfold, evaluate, ResultsManager
from optimizer import HC_GP, parse_grid, CSA, GA
from data import NTIDIGITS, FSDD, NMNIST, MITBIH

import sim_ballena as ballena
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# ============================
#       CONFIG DATASETS
# ============================

dataset_name = 'MITBIH'

if dataset_name  =='FSDD':		
	dataset      = FSDD(max_spikes=15)
	start_cell   = Cell(t_lims=(0,450), d2_lims=(0,39))
	n_epochs     = 60
	len_d3       = None
	len_vector   = 70
	only_d1 	 = False

elif dataset_name=='NTIDIGITS': 	
	dataset    = NTIDIGITS()
	start_cell = Cell(t_lims=(0,400), d2_lims=(0,64))
	n_epochs     = 60
	len_d3       = None
	len_vector   = 80
	only_d1 	 = False


elif dataset_name=='NMNIST':     
	dataset    = NMNIST()
	start_cell = Cell(t_lims=(0,350), d2_lims=(0,34), d3_lims=(0,34))
	n_epochs     = 60
	len_d3       = 34
	len_vector   = 80
	only_d1 	 = False

elif dataset_name=='MITBIH':    
	dataset    = MITBIH()
	start_cell = Cell(t_lims=(0,293), d2_lims=(0,2))
	n_epochs   = 35
	len_d3     = None
	len_vector = 70
	only_d1    = True

# ============================
#        LOAD DATASETS
# ============================

def NMNIST_2D(instance):
    return [ (t, convert_coords_to_2d(x,y,34)) for t,(x,y) in instance]

dataset.load()
t_train = dataset.t_train
t_test  = dataset.t_test
if dataset_name=='NMNIST':
	d_train = [ballena.Instance( NMNIST_2D(d) ) for d in dataset.d_train]
	d_test  = [ballena.Instance( NMNIST_2D(d) ) for d in dataset.d_test ]
else:
	d_train = [ballena.Instance(d) for d in dataset.d_train]
	d_test  = [ballena.Instance(d) for d in dataset.d_test ] 

# ============================
#        OTHER CONFIG
# ============================

sim_time = start_cell.t_lims[1]
mode     = start_cell.cell_type



In [3]:

# ============================
#        UTIL FUNCIONS
# ============================

def MITBIH_INITIAL(for_parser):
	children_is_null = False if for_parser else True
	initial_partition = Partition( SpaceType.D2 )
	initial_partition.assign( Splits.D2, 0.5, children_is_null=children_is_null )
	return initial_partition

# ==================
#   EVAL FUNCTIONS
# ==================
def objective(grid):
	global ml_model, dataset_name
	if dataset_name=='MITBIH':	metric='recall_mitbih'
	else					 :  metric='accuracy'
	net = generate_network(grid, sim_time, mode=start_cell.cell_type, len_d3=len_d3 )	
	s_train = [ net.simulate( d, sim_time ).spk_count() for d in d_train ]
	return -evaluate_kfold(s_train, t_train, model=ml_model, metric=metric)

def evaluate_final(grid, metric='accuracy'):
	global ml_model
	net = generate_network( grid, simulation_time=sim_time, mode=start_cell.cell_type, len_d3=len_d3 )
	s_train = [ net.simulate(instance,sim_time).spk_count() for instance in d_train ]
	s_test  = [ net.simulate(instance,sim_time).spk_count() for instance in d_test  ]
	return evaluate(s_train, s_test, t_train, t_test, model=ml_model, metric=metric), grid

def objective_parser(sol):
	if dataset_name=='MITBIH':	initial_partition=MITBIH_INITIAL(for_parser=True)
	else					 :  initial_partition=None
	grid = parse_grid( sol, start_cell, only_d1=only_d1, initial_partition=initial_partition )
	return objective(grid)

def evaluate_final_parser(sol):
	if dataset_name=='MITBIH':	initial_partition=MITBIH_INITIAL(for_parser=True)
	else					 :  initial_partition=None
	grid = parse_grid( sol, start_cell, only_d1=only_d1	, initial_partition=initial_partition)
	return evaluate_final(grid)

# ============================
#   ML MODELS CONFIGURATIONS
# ============================

readouts = [
	{
     'readout_name': 'LOGREG',
     'ml_model'	   : LogisticRegression(max_iter=500, solver='saga')
    },
    {
    'readout_name': 'RF',
    'ml_model'	  : RandomForestClassifier(n_estimators=30)
    },
    {
	'readout_name': 'SVM',
    'ml_model'	  : SVC( kernel='poly', degree=2 )
	},
    {
    'readout_name': 'KNN',
    'ml_model'	  : KNeighborsClassifier(n_neighbors=5, algorithm='ball_tree')
	}  
]


In [ ]:
# =====================
#     GA EXPERIMENT 
# =====================
results_manager = ResultsManager('results/lsmgrid_experiment_ga_aug10.json')

# ==================
#     OPTIMIZE
# ==================
for readout in readouts:
	readout_name, ml_model = readout.values()

	for trial_number in range(30):
		print('==================')
		print(f'   {readout_name} TRIAL {trial_number}')
		print('==================')

		ga = GA(objective_parser, 30, len_vector, 30, scale=0.04, n_mut=1, p_mut_frame=0.21, tournament_k=2)
		for epoch in range(n_epochs):
			ga.evolve()
			print(  f'iter {epoch+1}', ga.g_best.fit )

		history = [ s.fit for s in ga.history ]
		sol     = ga.g_best.sol
		final_acc, g = evaluate_final_parser( sol )
		g            = [ cell.to_dict() for cell in g ]
		trial = { 'history':history, 'best_grid':g, 'final_acc':final_acc }

		results_manager.add( f'{dataset_name}+{readout_name}', trial )

In [ ]:
# ===================
#  HC_GP EXPERIMENT 
# ===================
results_manager = ResultsManager('results/lsmgrid_experiment_hcgp_aug10.json')

# ==================
#     OPTIMIZE
# ==================
for readout in readouts:
	readout_name, ml_model = readout.values()
	
	for trial_number in range(30):
		print('==================')
		print(f'   {readout_name} TRIAL {trial_number}')
		print('==================')

		hcgp = HC_GP(objective, start_cell, n_process=30, n_children=30, only_d1=only_d1)
		if dataset_name=='MITBIH':
			hcgp.history = hcgp.evaluate( [MITBIH_INITIAL(for_parser=False)] )
			
		for e in range(n_epochs):
			hcgp.evolve(require_svc_start=(readout_name=='SVM' and dataset_name=='MITBIH'))
			print(e, hcgp.history[-1].fit)

		history      = [ trial.fit for trial in hcgp.history ]
		final_acc, g = evaluate_final( hcgp.history[-1].sol.get_grid(start_cell) )
		g            = [ cell.to_dict() for cell in g ]

		trial = { 'history':history, 'best_grid':g, 'final_acc':final_acc }
		results_manager.add( f'{dataset_name}+{readout_name}', trial )


In [ ]:

# ===================
#  CSA EXPERIMENT 
# ===================
results_manager = ResultsManager('results/lsmgrid_experiment_csa_aug10.json')

# ==================
#     OPTIMIZE
# ==================
for readout in readouts:
	readout_name, ml_model = readout.values()

	for trial_number in range(30):
		print('==================')
		print(f'   {readout_name} TRIAL {trial_number}')
		print('==================')

		csa = CSA(objective_parser, 30, len_vector, 30, AP=0.24, scale=0.05, n_mut=3, max_grace=1)
		for epoch in range(n_epochs):
			csa.evolve()
			print(  f'iter {epoch+1}', csa.g_best.fit  )

		# ==================
		#    SAVE STATS
		# ==================
		history = [ s.fit for s in csa.history ]
		sol     = csa.g_best.sol
		final_acc, g = evaluate_final_parser( sol )
		g            = [ cell.to_dict() for cell in g ]
		trial = { 'history':history, 'best_grid':g, 'final_acc':final_acc }

		results_manager.add( f'{dataset_name}+{readout_name}', trial )